<a href="https://colab.research.google.com/github/19mddill/Machine_Learning_Notebooks/blob/main/Dimensionality_Reduction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Principle *Components*

In [1]:
import numpy as np
from scipy.spatial.transform import Rotation

m = 60
X = np.zeros((m, 3))  # initialize 3D dataset
np.random.seed(42)
angles = (np.random.rand(m) ** 3 + 0.5) * 2 * np.pi  # uneven distribution
X[:, 0], X[:, 1] = np.cos(angles), np.sin(angles) * 0.5  # oval
X += 0.28 * np.random.randn(m, 3)  # add more noise
X = Rotation.from_rotvec([np.pi / 29, -np.pi / 20, np.pi / 4]).apply(X)
X += [0.2, 0, 0.2]  # shift a bit

In [2]:
X_centered = X - X.mean(axis=0)

In [3]:
U,s,Vt = np.linalg.svd(X_centered)
c1 = Vt[0]
c2 = Vt[1]
c1,c2

(array([0.67857588, 0.70073508, 0.22023881]),
 array([-0.72817329,  0.6811147 ,  0.07646185]))

In [4]:
w2 = Vt[:2].T

In [5]:
x2d = X_centered @ w2

In [6]:
from sklearn.decomposition import PCA
pca = PCA(n_components=2)
x2d = pca.fit_transform(X)

In [7]:
pca.components_

array([[ 0.67857588,  0.70073508,  0.22023881],
       [ 0.72817329, -0.6811147 , -0.07646185]])

In [8]:
pca.explained_variance_ratio_

array([0.7578477 , 0.15186921])

# Right Number of Dimensions

In [9]:
from sklearn.datasets import fetch_openml

In [10]:
mnist = fetch_openml('mnist_784', as_frame=False)

In [11]:
X_train,y_train = mnist['data'][:60000],mnist['target'][:60000]
X_test,y_test = mnist.data[60000:],mnist.target[60000:]

In [12]:
pca = PCA()
pca.fit(X_train)

PCA()

In [13]:
cumsum = np.cumsum(pca.explained_variance_ratio_)
d = np.argmax(cumsum >= 0.95) + 1

In [14]:
d

np.int64(154)

In [15]:
pca = PCA(n_components=0.95)
X_reduced = pca.fit_transform(X_train)

In [16]:
pca.n_components_

np.int64(154)

In [17]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV
from sklearn.pipeline import make_pipeline

In [18]:
clf = make_pipeline(PCA(random_state=42),RandomForestClassifier(random_state=42))

In [19]:
param_distrib = {
    "pca__n_components": np.arange(10, 80),
    "randomforestclassifier__n_estimators": np.arange(50, 500)
}

In [20]:
rnd_search = RandomizedSearchCV(clf, param_distrib, n_iter=10, cv=3,
                                random_state=42)
rnd_search.fit(X_train[:1000], y_train[:1000])

RandomizedSearchCV(cv=3,
                   estimator=Pipeline(steps=[('pca', PCA(random_state=42)),
                                             ('randomforestclassifier',
                                              RandomForestClassifier(random_state=42))]),
                   param_distributions={'pca__n_components': array([10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26,
       27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43,
       44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60,
       6...
       414, 415, 416, 417, 418, 419, 420, 421, 422, 423, 424, 425, 426,
       427, 428, 429, 430, 431, 432, 433, 434, 435, 436, 437, 438, 439,
       440, 441, 442, 443, 444, 445, 446, 447, 448, 449, 450, 451, 452,
       453, 454, 455, 456, 457, 458, 459, 460, 461, 462, 463, 464, 465,
       466, 467, 468, 469, 470, 471, 472, 473, 474, 475, 476, 477, 478,
       479, 480, 481, 482, 483, 484, 485, 486, 487, 488, 489, 490, 491,
       492, 493, 494, 495, 496, 497, 498, 499])},
                   random_state=42)

In [21]:
print(rnd_search.best_params_)

{'randomforestclassifier__n_estimators': np.int64(475), 'pca__n_components': np.int64(57)}


# Incremental PCA

In [27]:
from sklearn.decomposition import IncrementalPCA

n_batches = 100
inc_pca = IncrementalPCA(n_components=154)
for X_batch in np.array_split(X_train, n_batches):
    inc_pca.partial_fit(X_batch)

X_reduced = inc_pca.transform(X_train)

# Memory mapped

In [28]:
filename = "my_mnist.mmap"

In [29]:
X_mmap = np.memmap(filename,dtype='float32',mode='write',shape=X_train.shape)
X_mmap[:] = X_train
X_mmap.flush() # any data still in the cache gets saved to the disk

In [30]:
X_mmap = np.memmap(filename,dtype='float32',mode='readonly').reshape(-1,784)
batch_size = X_mmap.shape[0] // n_batches
inc_pca = IncrementalPCA(n_components=154,batch_size=batch_size)
inc_pca.fit(X_mmap)

IncrementalPCA(batch_size=600, n_components=154)

# Random Projection

In [8]:
from sklearn.random_projection import johnson_lindenstrauss_min_dim
import numpy as np

m,epsilon = 5_000,0.1
d = johnson_lindenstrauss_min_dim(m,eps=epsilon)
d

np.int64(7300)

In [11]:
n = 20_000
np.random.seed(42)
P = np.random.randn(d,n)/np.sqrt(d)
X = np.random.randn(m,n)
X_reduced = X @ P.T

In [13]:
from sklearn.random_projection import GaussianRandomProjection

gaussian_rnd_proj = GaussianRandomProjection(eps = 0.1, random_state = 42)
X_reduced = gaussian_rnd_proj.fit_transform(X)

In [14]:
gaussian_rnd_proj.components_

array([[ 5.81359943e-03, -1.61826124e-03,  7.58062095e-03, ...,
         4.42829826e-03,  2.00553485e-02, -1.89597272e-02],
       [ 4.07638220e-03,  3.31605183e-03, -1.09611357e-02, ...,
         7.21559891e-03,  9.53444696e-03,  4.16636507e-03],
       [-5.52267488e-03,  1.18527859e-02, -2.31960160e-03, ...,
         1.65039881e-03, -2.55380604e-02, -7.48780800e-05],
       ...,
       [-1.03564530e-02,  7.94509570e-03, -5.69474166e-03, ...,
         1.14580829e-02, -1.29950377e-02, -1.59301221e-02],
       [-6.77138686e-03, -5.90454856e-03,  1.23942811e-02, ...,
        -2.01775486e-02, -1.75452980e-03, -7.52724235e-03],
       [-4.12988138e-03,  1.24015654e-02,  3.58943880e-04, ...,
        -6.18872802e-03,  5.77292951e-04,  6.83212442e-03]])

# Sparse Random Projection

In [15]:
from sklearn.random_projection import johnson_lindenstrauss_min_dim
import numpy as np

m, epsilon = 5_000, 0.1
d = johnson_lindenstrauss_min_dim(m, eps=epsilon)
n = 20_000
np.random.seed(42)



In [16]:
# calculate density and v
r = 1 / np.sqrt(n)
v = 1 / np.sqrt(d * r)

# build sparse matrix P manually
random_draws = np.random.rand(d, n)   # uniform random values between 0 and 1
P = np.zeros((d, n))                  # start with all zeros

P[random_draws < r/2]  = +v           # probability r/2 → assign +v
P[random_draws > 1-r/2] = -v          # probability r/2 → assign -v
                                       # rest stays 0    → probability 1-r



In [17]:
# fake dataset
X = np.random.randn(m, n)

# project
X_reduced = X @ P.T

print(f"Original shape:  {X.shape}")         # (5000, 20000)
print(f"Reduced shape:   {X_reduced.shape}")  # (5000, d)
print(f"Density (r):     {r:.4f}")            # ≈ 0.007
print(f"Value (v):       {v:.4f}")            # ≈ 1/sqrt(d*r)
print(f"Non-zero ratio:  {np.mean(P != 0):.4f}")  # should be ≈ r

Original shape:  (5000, 20000)
Reduced shape:   (5000, 7300)
Density (r):     0.0071
Value (v):       0.1392
Non-zero ratio:  0.0071


# Inverse

In [18]:
component_pinv = np.linalg.pinv(gaussian_rnd_proj.components_)

In [20]:
X_recoverd = X_reduced @ component_pinv.T